# Figuras y cifras del capítulo 5 — Detección y evaluación del modelo (OE2)

Todo lo que el capítulo cita sale de aquí: nada de números a mano en el `.tex`.
El cuaderno escribe cuatro cosas en `../figures/`:

| Salida | Qué es | Cómo lo usa el `.tex` |
|---|---|---|
| `valores.tex` | macros con cada cifra (`\mQueries`, `\rRecallYol`, …) | `\input` en el preámbulo |
| `tablas/*.tex` | las filas de cada tabla, sin cabecera ni `\bottomrule` | `\input{…} \\` dentro del `tabular` |
| `data/*.dat` | series para pgfplots | `\addplot table` |
| `*.png` | las figuras que son **imagen** o nube de cientos de puntos | `\includegraphics` |

**Qué va en matplotlib y qué va en LaTeX.** Los **esquemas** (arquitectura, pirámide,
capa del decodificador, muestreo deformable, emparejamiento, pipeline, flujo de
inferencia) se dibujan en **TikZ** dentro del `.tex`; las **curvas y los histogramas**
(mAP por época, cobertura frente a carga, barrido del umbral, distribución del IoU)
en **pgfplots** desde los `.dat`. Sólo son PNG la matriz de confusión (25x25 celdas),
los espectrogramas cualitativos y la demostración sobre una grabación.

Fuentes: `runs/<corrida>/` (config.json, train.log, metrics.jsonl, best.pt, volcados
`*_predictions.pt`), `runs/comparacion/`, `data/processed/` y `data/cleaned/`.


In [1]:
import json
import re
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

NB_DIR = Path.cwd()
CHAPTER_DIR = NB_DIR.parent
PROJECT_DIR = next(p for p in (NB_DIR, *NB_DIR.parents) if (p / "pyproject.toml").exists())
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

FIG = CHAPTER_DIR / "figures"
RUTA_FIG = "capitulos/05_oe2_detector/figures"
TAB = FIG / "tablas"
DAT = FIG / "data"
for path in (FIG, TAB, DAT):
    path.mkdir(parents=True, exist_ok=True)

TEXT_WIDTH_IN = 6.1
DPI = 200
AZUL, CELESTE, VERDE, ROSA, AMBAR, GRIS = (
    "#1E40AF",
    "#00B4D8",
    "#22C55E",
    "#FF5FA2",
    "#F59E0B",
    "#6B7280",
)
mpl.rcParams.update(
    {
        "figure.dpi": 100,
        "savefig.dpi": DPI,
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def number(value, decimals: int = 0) -> str:
    """Cifra a la española: miles con espacio fino y coma decimal."""
    if decimals == 0:
        return f"{round(value):,}".replace(",", "\\,")
    whole, fraction = f"{value:,.{decimals}f}".split(".")
    return whole.replace(",", "\\,") + "," + fraction


def percent(value: float, decimals: int = 1) -> str:
    return number(100 * value, decimals) + "\\,\\%"


def macro_name(key: str) -> str:
    words = key.replace("-", "_").split("_")
    name = words[0] + "".join(w.capitalize() for w in words[1:])
    return "".join(c for c in name if c.isalpha())


VALORES: dict[str, str] = {}


def put(**values) -> None:
    """Acumula macros; se escriben todas juntas al final (`escribir_macros`)."""
    for key, value in values.items():
        VALORES[key] = value


def escribir_macros() -> None:
    # Los dos capítulos cargan su valores.tex en el mismo preámbulo: un nombre repetido
    # haría fallar la compilación con "command already defined". Se comprueba acá.
    otros = Path(CHAPTER_DIR.parent / "04_oe1_curacion/figures/valores.tex")
    usados = set(re.findall(r"\\newcommand\{\\([A-Za-z]+)\}", otros.read_text()))
    choques = sorted(usados & {macro_name(k) for k in VALORES})
    assert not choques, f"macros repetidas con el capítulo 4: {choques}"
    lines = [f"\\newcommand{{\\{macro_name(k)}}}{{{v}}}" for k, v in sorted(VALORES.items())]
    (FIG / "valores.tex").write_text(
        "% generado por notebook/figuras_05_oe2_detector.ipynb; no editar\n"
        + "\n".join(lines)
        + "\n"
    )
    print(f"{len(lines)} macros -> figures/valores.tex")


def escribir_tabla(nombre: str, filas: list[str]) -> None:
    (TAB / f"{nombre}.tex").write_text(
        "% generado por el cuaderno; no editar\n" + " \\\\\n".join(filas) + "\n"
    )
    print(f"tablas/{nombre}.tex: {len(filas)} filas")


def escribir_dat(nombre: str, tabla: pd.DataFrame) -> None:
    tabla.to_csv(DAT / f"{nombre}.dat", sep=" ", index=False, float_format="%.6g")
    print(f"data/{nombre}.dat: {tabla.shape[0]}x{tabla.shape[1]}")


def guardar(fig, nombre: str) -> None:
    fig.savefig(FIG / f"{nombre}.png", bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    print(f"{nombre}.png")


# Un identificador más largo que esto no cabe en un renglón y, sin punto de corte,
# desborda el margen; uno corto (una clase como `sb/spc`) no debe partirse nunca.
LARGO_PARTIBLE = 16


def tt(texto: str) -> str:
    escapado = texto.replace("_", "\\_")
    if len(texto) > LARGO_PARTIBLE:
        escapado = escapado.replace("/", "/\\allowbreak ").replace("-", "-\\allowbreak ")
    return "\\code{" + escapado + "}"


## 1. RE2.1 — la arquitectura tal como la declara el código

Los hiperparámetros no se transcriben: se leen de los módulos de `src/models/` y del
`config.json` de la corrida, de modo que una cifra del capítulo no puede desviarse de
la que se entrenó sin que el cuaderno lo note.


In [2]:
from core.config import MAX_DETECTIONS, SCORE_FLOOR, SEED, P  # noqa: E402
from models import criterion as crit  # noqa: E402
from models import deformable_detr as dd  # noqa: E402
from models.backbone import AST_CHECKPOINT, N_LEVELS, TIME_STRIDE  # noqa: E402
from models.coco_deformable_detr import COCO_QUERIES, DETR_CHECKPOINT  # noqa: E402
from models.faster_rcnn import ANCHOR_RATIOS, MAX_SIZE, MIN_SIZE, TRAINABLE_LAYERS  # noqa: E402
from models.yolo import DEFAULT_MODEL, IMAGE_SIZE  # noqa: E402

RUNS = PROJECT_DIR / "runs"
# La corrida de cada configuración de la comparación vigente, con el nombre corto con el
# que la nombran las macros (`\rRecallDet`, `\cHorasYol`, ...).
CORRIDAS = {
    "Det": "detr_t10_logmel_birds_v3",
    "Frc": "frcnn_v3",
    "Yol": "yolo26s_v3",
    "Rtd": "rtdetr_l_v3",
}
NOMBRE = {
    "Det": "AST-Def-DETR (propuesta)",
    "Frc": "Faster R-CNN R50-FPN v2",
    "Yol": "YOLO26s",
    "Rtd": "RT-DETR-L",
}
ORDEN = ["Det", "Frc", "Yol", "Rtd"]

config = {k: json.loads((RUNS / v / "config.json").read_text()) for k, v in CORRIDAS.items()}
hp = config["Det"]["hparams"]
assert hp["dim"] == dd.DIM and hp["n_queries"] == dd.N_QUERIES, "la corrida no usó los defaults"

put(
    m_ast_checkpoint=tt(AST_CHECKPOINT),
    m_time_stride=str(TIME_STRIDE),
    m_dim=str(dd.DIM),
    m_queries=str(dd.N_QUERIES),
    m_levels=str(N_LEVELS),
    m_decoder_layers=str(dd.N_DECODER_LAYERS),
    m_heads=str(dd.N_HEADS),
    m_points=str(dd.N_POINTS),
    m_ffn=number(dd.FFN),
    m_dropout=number(dd.DROPOUT, 1),
    m_prior=number(dd.PRIOR_PROB, 2),
    m_focal_alpha=number(crit.FOCAL_ALPHA, 2),
    m_focal_gamma=number(crit.FOCAL_GAMMA, 1),
    m_cost_class=number(crit.COST_CLASS, 1),
    m_cost_bbox=number(crit.COST_BBOX, 1),
    m_cost_iou=number(crit.COST_IOU, 1),
    m_weight_class=number(crit.WEIGHT_CLASS, 1),
    m_nms_iou=number(config["Det"]["nms_iou"], 1),
    m_score_floor=number(SCORE_FLOOR, 3),
    m_max_det=number(MAX_DETECTIONS),
    m_frontend=tt(hp["frontend"]),
)
print(hp)


{'n_frames': 331, 'time_stride': 10, 'dim': 128, 'n_queries': 100, 'frontend': 'logmel'}


### Tokens del AST y resolución de cada nivel de la pirámide

El AST corta el mel en parches de 16x16 con paso `(frequency_stride, time_stride)`.
Con 128 bandas y `n_frames` cuadros quedan `freq_out x time_out` tokens; la pirámide
los lleva a cuatro mapas. Lo que importa para el capítulo es cuántos milisegundos de
la ventana y cuántas bandas mel cubre una celda de cada nivel.


In [3]:
PATCH, FREQ_STRIDE = 16, 10
freq_out = (P.n_mels - PATCH) // FREQ_STRIDE + 1
time_out = (hp["n_frames"] - PATCH) // hp["time_stride"] + 1
ESCALAS = [("$4\\times$", 4), ("$2\\times$", 2), ("$1\\times$", 1), ("$\\tfrac{1}{2}\\times$", 0.5)]

filas_niveles, ms_por_celda, bandas_por_celda = [], [], []
for etiqueta, factor in ESCALAS:
    alto, ancho = int(freq_out * factor), int(time_out * factor)
    ms = 1000 * P.clip_len_s / ancho
    bandas = P.n_mels / alto
    ms_por_celda.append(ms)
    bandas_por_celda.append(bandas)
    filas_niveles.append(
        f"{etiqueta} & ${alto} \\times {ancho}$ & {number(ms, 1)} & {number(bandas, 1)}"
    )
escribir_tabla("niveles", filas_niveles)

put(
    m_tokens_freq=str(freq_out),
    m_tokens_time=str(time_out),
    m_token_ms=number(1000 * P.clip_len_s / time_out, 0),
    m_cell_zero=number(ms_por_celda[0], 1),
    m_cell_one=number(ms_por_celda[1], 1),
    m_cell_two=number(ms_por_celda[2], 1),
    m_cell_three=number(ms_por_celda[3], 1),
    m_cell_zero_plain=f"{ms_por_celda[0]:.1f}",
    m_cell_one_plain=f"{ms_por_celda[1]:.1f}",
    m_cell_two_plain=f"{ms_por_celda[2]:.1f}",
    m_cell_three_plain=f"{ms_por_celda[3]:.1f}",
)
print(f"tokens {freq_out} x {time_out} | ms/celda {[round(v, 1) for v in ms_por_celda]}")


tablas/niveles.tex: 4 filas
tokens 12 x 32 | ms/celda [23.4, 46.9, 93.8, 187.5]


### Parámetros por bloque

Se cuentan sobre el `state_dict` del `best.pt`, que es el archivo que se publica: así
el reparto que declara el capítulo es el del modelo que se entrega, no el de una
instancia recién construida.


In [4]:
BLOQUES = [
    ("Extractor AST ({ast} capas, {hidden}\\,d)", ("backbone.",)),
    ("Front-end y normalización de entrada", ("frontend.", "input_norm.")),
    ("Proyección a la dimensión de la cabeza", ("head.proj.",)),
    ("Pirámide multiescala", ("head.pyramid.",)),
    ("Decodificador deformable ({capas} capas)", ("head.detr.",)),
]

ck = torch.load(RUNS / CORRIDAS["Det"] / "best.pt", map_location="cpu", weights_only=False)
sd = ck["state_dict"]
hidden = sd["backbone.model.embeddings.cls_token"].shape[-1]
n_ast_layers = 1 + max(int(k.split(".")[3]) for k in sd if k.startswith("backbone.model.layers."))
total = sum(t.numel() for t in sd.values())

filas_bloques = []
por_bloque = {}
for plantilla, prefijos in BLOQUES:
    n = sum(t.numel() for k, t in sd.items() if k.startswith(prefijos))
    por_bloque[plantilla] = n
    nombre = plantilla.format(hidden=hidden, capas=dd.N_DECODER_LAYERS, ast=n_ast_layers)
    filas_bloques.append(f"{nombre} & {number(n / 1e6, 2)} & {percent(n / total)}")
filas_bloques.append(
    f"\\textbf{{Total}} & \\textbf{{{number(total / 1e6, 2)}}} & \\textbf{{100\\,\\%}}"
)
escribir_tabla("bloques", filas_bloques)

cabeza = total - por_bloque[BLOQUES[0][0]] - por_bloque[BLOQUES[1][0]]
put(
    m_hidden=str(hidden),
    m_ast_layers=str(n_ast_layers),
    m_params=number(total / 1e6, 1),
    m_params_head=number(cabeza / 1e6, 1),
    m_params_backbone_pct=percent(por_bloque[BLOQUES[0][0]] / total),
    m_tensors=number(len(sd)),
    m_best_epoch=str(ck["epoch"] + 1),
    m_checkpoint_mb=number((RUNS / CORRIDAS["Det"] / "best.pt").stat().st_size / 1e6),
)
print(f"{total / 1e6:.2f}M total | cabeza {cabeza / 1e6:.2f}M | {len(sd)} tensores")


tablas/bloques.tex: 6 filas
88.64M total | cabeza 3.08M | 398 tensores


### Duración de cada clase frente a la resolución de la pirámide

La pirámide existe porque las clases no ocupan una sola escala. La serie que sigue
alimenta la figura de pgfplots que sitúa las 25 clases (mediana y rango
intercuartílico de la duración) junto a las cuatro resoluciones y a la ventana.


In [5]:
from core.config import CLEANED_DIR  # noqa: E402
from data.annotations import load_annotations  # noqa: E402
from prepare_data import select_experiment  # noqa: E402

anotaciones = load_annotations(CLEANED_DIR)
experimento, etiquetas = select_experiment(anotaciones)
CLASES = list(etiquetas.names)
assert CLASES == list(json.loads((PROJECT_DIR / "data/processed/labels.json").read_text()).values())

dur = experimento.assign(ms=1000 * experimento["duration_s"]).groupby("label")["ms"]
duraciones = (
    pd.DataFrame(
        {
            "clase": dur.median().index,
            "med": dur.median().to_numpy(),
            "q1": dur.quantile(0.25).to_numpy(),
            "q3": dur.quantile(0.75).to_numpy(),
        }
    )
    .sort_values("med")
    .reset_index(drop=True)
)
escribir_dat("duraciones", duraciones)

bajo_nivel_fino = int((duraciones["med"] < ms_por_celda[0]).sum())
sobre_ventana = int((duraciones["med"] > 1000 * P.clip_len_s).sum())
put(
    m_classes_below_cell=str(bajo_nivel_fino),
    m_classes_over_window=str(sobre_ventana),
    m_shortest_class=tt(duraciones.iloc[0]["clase"]),
    m_shortest_class_ms=number(duraciones.iloc[0]["med"], 0),
    m_longest_class=tt(duraciones.iloc[-1]["clase"]),
    m_longest_class_s=number(duraciones.iloc[-1]["med"] / 1000, 1),
)
print(duraciones.head(3).to_string(), "\n...\n", duraciones.tail(2).to_string())


data/duraciones.dat: 25x4
    clase         med          q1          q3
0   sm/cc   81.203064   63.940123  106.783595
1   aa/gc   93.125866   76.942777  118.238600
2  sb/spc  161.727773  127.206335  220.610381 
...
     clase           med           q1            q3
23  pt/dc   6671.144707  4756.023016   9549.275578
24  as/hc  10993.365272  7128.876548  15807.443681


## 2. Las configuraciones comparadas y sus recetas

Parámetros y época elegida salen del `train.log` de cada corrida (las dos del
proyecto) o del resumen de Ultralytics en `nohup.log` (las dos suyas). El costo en
horas de GPU es la diferencia entre la primera y la última marca de tiempo del
registro, de modo que incluye la validación de cada época.


In [6]:
MARCA = re.compile(r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]")
PARAMS_PROPIO = re.compile(r"\|\s*([\d.]+)M parámetros \(([\d.]+)M entrenables\)")
PARAMS_ULTRA = re.compile(r"summary: \d+ layers, ([\d,]+) parameters")
EPOCAS_ULTRA = re.compile(r"(\d+) epochs completed in ([\d.]+) hours")
MEJOR = re.compile(r"nuevo mejor mAP30=([\d.]+)")


def marcas(texto: str) -> list[datetime]:
    return [datetime.strptime(m, "%Y-%m-%d %H:%M:%S") for m in MARCA.findall(texto)]


def leer_corrida(clave: str) -> dict:
    d = RUNS / CORRIDAS[clave]
    log = (d / "train.log").read_text()
    ultra = (d / "nohup.log").read_text() if config[clave]["architecture"] == "yolo" else ""
    info: dict = {"nombre": CORRIDAS[clave], "epocas_plan": config[clave]["epochs"]}
    if ultra:
        info["params"] = int(PARAMS_ULTRA.search(ultra).group(1).replace(",", "")) / 1e6
        info["entrenables"] = info["params"]
        epocas, horas = EPOCAS_ULTRA.search(ultra).groups()
        info["epocas"] = int(epocas)
        info["horas"] = float(horas)
        curva = pd.read_csv(d / "results.csv")
        columna = next(c for c in curva.columns if "mAP50(B)" in c)
        info["mejor_epoca"] = int(curva[columna].idxmax()) + 1
        info["criterio"] = "aptitud de Ultralytics"
    else:
        totales, entrenables = PARAMS_PROPIO.search(log).groups()
        info["params"], info["entrenables"] = float(totales), float(entrenables)
        epocas_log = [datetime.strptime(m, "%Y-%m-%d %H:%M:%S") for m in MARCA.findall(log)]
        info["epocas"] = len(re.findall(r"\[\s*\d+/\d+\] loss=", log))
        info["horas"] = (epocas_log[-1] - epocas_log[0]).total_seconds() / 3600
        metricas = [json.loads(line) for line in (d / "metrics.jsonl").read_text().splitlines()]
        info["mejor_epoca"] = 1 + max(range(len(metricas)), key=lambda i: metricas[i]["val"]["map_30"])
        info["criterio"] = "mAP@0,3 en validación"
    info["min_epoca"] = 60 * info["horas"] / info["epocas"]
    return info


corridas = {k: leer_corrida(k) for k in ORDEN}
for k, v in corridas.items():
    print(f"{k:4s} {v['nombre']:28s} {v['params']:6.1f}M  {v['epocas']:3d} ép  "
          f"mejor {v['mejor_epoca']:3d}  {v['horas']:5.2f} h")

filas_costo = []
for k in ORDEN:
    c = corridas[k]
    filas_costo.append(
        f"{NOMBRE[k]} & {c['epocas']} de {c['epocas_plan']} & {c['mejor_epoca']} & "
        f"{number(c['min_epoca'], 1)} & {number(c['horas'], 1)}"
    )
escribir_tabla("costo", filas_costo)

for k in ORDEN:
    c = corridas[k]
    put(**{
        f"c_params_{k.lower()}": number(c["params"], 1),
        f"c_trainable_{k.lower()}": number(c["entrenables"], 1),
        f"c_epochs_{k.lower()}": str(c["epocas"]),
        f"c_best_epoch_{k.lower()}": str(c["mejor_epoca"]),
        f"c_hours_{k.lower()}": number(c["horas"], 1),
    })
put(c_hours_total=number(sum(c["horas"] for c in corridas.values()), 1))


Det  detr_t10_logmel_birds_v3       88.6M   30 ép  mejor  16   6.60 h
Frc  frcnn_v3                       43.4M   12 ép  mejor   7   5.61 h
Yol  yolo26s_v3                     10.0M   27 ép  mejor  17   1.02 h
Rtd  rtdetr_l_v3                    32.9M   23 ép  mejor  13   4.18 h
tablas/costo.tex: 4 filas


### Configuraciones comparadas, recetas y claves del checkpoint


In [7]:
QUE_CONTROLA = {
    "Det": "---",
    "Frc": "detector de dos etapas preentrenado en COCO",
    "Yol": "detector denso de una etapa preentrenado en COCO",
    "Rtd": "cabeza de conjuntos con extractor de imagen preentrenado en COCO",
}
EXTRACTOR = {
    "Det": "AST / AudioSet (afinado); cabeza / aves",
    "Frc": "ResNet-50 / COCO",
    "Yol": "CSP / COCO",
    "Rtd": "HGNetv2 / COCO",
}
PREENTRENADO = {
    "Det": "extractor y cabeza",
    "Frc": "detector entero",
    "Yol": "detector entero",
    "Rtd": "detector entero",
}

filas_config = []
for k in ORDEN:
    c = corridas[k]
    filas_config.append(
        f"{NOMBRE[k]} & {EXTRACTOR[k]} & {number(c['params'], 1)} & {PREENTRENADO[k]} & "
        f"{QUE_CONTROLA[k]}"
    )
escribir_tabla("configuraciones", filas_config)

filas_recetas = []
for k in ORDEN:
    cfg, c = config[k], corridas[k]
    if cfg["architecture"] == "yolo":
        lote, lr = cfg["batch_size"], cfg.get("lr0")
        optimizador = cfg.get("optimizer", "auto (Ultralytics)")
        lr_txt = "auto" if lr is None else f"$10^{{{round(np.log10(lr))}}}$"
    else:
        lote, optimizador = cfg["batch_size"], "AdamW + OneCycle"
        lr_txt = f"$10^{{{round(np.log10(cfg['learning_rate']))}}}$"
    filas_recetas.append(
        f"{NOMBRE[k]} & {cfg['epochs']} & {lote} & {lr_txt} & {optimizador} & {c['criterio']}"
    )
escribir_tabla("recetas", filas_recetas)

CLAVES = {
    "architecture": "clave de \\code{models.registry.ARCHITECTURES} con la que se rearma",
    "hparams": "hiperparámetros del modelo, los mismos que recibió \\code{build\\_model}",
    "labels": "las clases, en orden: fija el número de salidas y el nombre de cada una",
    "config": "la receta de la corrida y el \\file{meta.json} del caché con el que se entrenó",
    "state_dict": "los pesos",
    "epoch": "la época de la que salió",
    "metrics": "sus métricas de validación",
}
filas_ck = []
for clave, que in CLAVES.items():
    valor = ck[clave]
    if clave == "state_dict":
        resumen = f"{number(len(valor))} tensores, {number(total / 1e6, 1)}\\,M de parámetros"
    elif clave == "labels":
        resumen = f"{len(valor)} nombres"
    elif isinstance(valor, dict):
        resumen = f"{len(valor)} claves"
    else:
        resumen = tt(str(valor)) if clave == "architecture" else str(valor + 1)
    filas_ck.append(f"{tt(clave)} & {resumen} & {que}")
escribir_tabla("checkpoint", filas_ck)


tablas/configuraciones.tex: 4 filas
tablas/recetas.tex: 4 filas
tablas/checkpoint.tex: 7 filas


### Curvas de entrenamiento

Una serie por corrida con el mAP@0,3 de validación por época. Las dos corridas del
proyecto lo escriben en `metrics.jsonl`; las de Ultralytics, en `results.csv` con su
propio mAP@0,5, que no es la misma magnitud y por eso va en un eje aparte.


In [8]:
def curva_map(clave: str) -> pd.Series:
    d = RUNS / CORRIDAS[clave]
    if config[clave]["architecture"] == "yolo":
        tabla = pd.read_csv(d / "results.csv")
        columna = next(c for c in tabla.columns if "mAP50(B)" in c)
        return pd.Series(tabla[columna].to_numpy(), index=range(1, len(tabla) + 1))
    filas = [json.loads(line) for line in (d / "metrics.jsonl").read_text().splitlines()]
    return pd.Series([f["val"]["map_30"] for f in filas], index=[f["epoch"] for f in filas])


# Un archivo por corrida: las cuatro tienen distinto número de épocas y pgfplots no
# necesita rellenar huecos.
for k in ORDEN:
    serie = curva_map(k)
    escribir_dat(
        f"curva_{k.lower()}",
        pd.DataFrame({"epoca": serie.index, "map": serie.to_numpy()}),
    )
    print(f"{k}: {len(serie)} épocas, máximo {serie.max():.3f} en la {serie.idxmax()}")


data/curva_det.dat: 30x2
Det: 30 épocas, máximo 0.632 en la 16
data/curva_frc.dat: 12x2
Frc: 12 épocas, máximo 0.596 en la 7
data/curva_yol.dat: 27x2
Yol: 27 épocas, máximo 0.596 en la 17
data/curva_rtd.dat: 23x2
Rtd: 23 épocas, máximo 0.576 en la 13


## 3. RE2.3 — la comparación

Se cargan los volcados crudos (`<corrida>_<split>_predictions.pt`: todas las cajas por
encima de 0,001, con las anotadas y el mapa ventana → grabación) y se les aplica el
protocolo de `src/evaluation/protocol.py`. El cuaderno no recalcula el protocolo a su
manera: importa el mismo módulo que usa `compare_models.py`, y comprueba al final que
las cifras coinciden con las de `runs/comparacion/comparacion_modelos_v3.json`.


In [9]:
from evaluation.evaluator import RawPredictions  # noqa: E402
from evaluation.metrics import (  # noqa: E402
    MATCH_IOU,
    WINDOW_CLASS_WIDTH,
    average_precision_per_class,
    mean_average_precision,
    window_classes,
)
from evaluation.protocol import (  # noqa: E402
    MIN_PRECISIONS,
    N_BOOTSTRAP,
    STRICT_IOU,
    bootstrap,
    equalize,
    match,
    measure_axes,
    operating_point,
    select,
)

LOOSE_IOU = 0.25  # la cota laxa que pide el IOV de RE2.3, junto a la estricta de 0,5

volcados = {
    (k, split): RawPredictions.load(RUNS / v / f"{v}_{split}_predictions.pt")
    for k, v in CORRIDAS.items()
    for split in ("val", "test")
}
ref = volcados[("Det", "test")]
NOMBRES_CLASE = ref.labels
assert NOMBRES_CLASE == CLASES
for (k, split), d in volcados.items():
    assert d.labels == NOMBRES_CLASE and d.n_images == volcados[("Det", split)].n_images

VENTANA = window_classes(ref.truth, len(NOMBRES_CLASE))
DETECCION = [c for c in range(len(NOMBRES_CLASE)) if c not in VENTANA]
HORAS_TEST = ref.n_images * P.clip_hop_s / 3600

put(
    p_match_iou=number(MATCH_IOU, 1),
    p_strict_iou=number(STRICT_IOU, 1),
    p_loose_iou=number(LOOSE_IOU, 2),
    p_min_precision=number(MIN_PRECISIONS[0], 2),
    p_min_precision_alt=number(MIN_PRECISIONS[1], 2),
    p_bootstrap=number(N_BOOTSTRAP),
    p_window_width=percent(WINDOW_CLASS_WIDTH, 0),
    p_seed=str(SEED),
    r_window_classes=", ".join(tt(NOMBRES_CLASE[c]) for c in VENTANA),
    r_n_detection_classes=str(len(DETECCION)),
    r_hours_test=number(HORAS_TEST, 2),
)
print(f"clases de ventana: {[NOMBRES_CLASE[c] for c in VENTANA]} | test {HORAS_TEST:.2f} h")


clases de ventana: ['as/hc', 'pt/dc'] | test 3.17 h


In [10]:
def medir(clave: str) -> dict:
    val = equalize(volcados[(clave, "val")].predictions, MAX_DETECTIONS)
    test = equalize(volcados[(clave, "test")].predictions, MAX_DETECTIONS)
    d_val, d_test = volcados[(clave, "val")], volcados[(clave, "test")]
    emparejado_val = match(val, d_val.truth, d_val.n_images)
    emparejado_test = match(test, d_test.truth, d_test.n_images)
    elegido = select(emparejado_val, MIN_PRECISIONS[0])
    punto = operating_point(emparejado_test, elegido.threshold)
    ic = bootstrap(emparejado_test, d_test.recordings.to(torch.int64), elegido.threshold)
    ejes = measure_axes(test, d_test.truth, NOMBRES_CLASE, elegido.threshold)
    ap = {
        iou: average_precision_per_class(test, d_test.truth, len(NOMBRES_CLASE), iou)
        for iou in (LOOSE_IOU, MATCH_IOU, STRICT_IOU)
    }
    f1 = 2 * punto.recall * punto.precision / (punto.recall + punto.precision)
    return {
        "umbral": elegido.threshold,
        "val": elegido,
        "test": punto,
        "f1": f1,
        "ic": ic,
        "ejes": ejes,
        "ap": ap,
        "map": {iou: mean_average_precision(a, DETECCION) for iou, a in ap.items()},
        "propuestas_hora": punto.n_above / HORAS_TEST,
        "fp_tp": (punto.n_above - punto.n_tp) / punto.n_tp,
        "val_boxes": val,
        "test_boxes": test,
        "emparejado_test": emparejado_test,
    }


t0 = time.time()
R = {k: medir(k) for k in ORDEN}
print(f"protocolo aplicado en {time.time() - t0:.0f} s")

# Control: las cifras que el protocolo ya escribió en runs/comparacion/ deben coincidir.
oficial = json.loads((RUNS / "comparacion/comparacion_modelos_v3.json").read_text())
por_modelo = {m["model"]: m for m in oficial["models"]}
for k in ORDEN:
    o = por_modelo[CORRIDAS[k]]
    assert abs(o["test"]["recall"] - R[k]["test"].recall) < 1e-9, k
    assert abs(o["map_30"] - R[k]["map"][MATCH_IOU]) < 1e-9, k
    assert abs(o["map_50"] - R[k]["map"][STRICT_IOU]) < 1e-9, k
print("coincide con runs/comparacion/comparacion_modelos_v3.json")

for k in ORDEN:
    r = R[k]
    print(
        f"{k:4s} umbral {r['umbral']:.2f} | recall {r['test'].recall:.3f} "
        f"[{r['ic']['recall'][0]:.3f}, {r['ic']['recall'][1]:.3f}] | "
        f"prec {r['test'].precision:.3f} | F1 {r['f1']:.3f} | "
        f"mAP@0,3 {r['map'][MATCH_IOU]:.3f} | mAP@0,5 {r['map'][STRICT_IOU]:.3f} | "
        f"{r['propuestas_hora']:.0f} prop/h"
    )


protocolo aplicado en 32 s
coincide con runs/comparacion/comparacion_modelos_v3.json
Det  umbral 0.21 | recall 0.797 [0.771, 0.825] | prec 0.669 | F1 0.728 | mAP@0,3 0.577 | mAP@0,5 0.451 | 2984 prop/h
Frc  umbral 0.69 | recall 0.781 [0.753, 0.808] | prec 0.680 | F1 0.727 | mAP@0,3 0.574 | mAP@0,5 0.457 | 2877 prop/h
Yol  umbral 0.11 | recall 0.811 [0.787, 0.835] | prec 0.665 | F1 0.731 | mAP@0,3 0.620 | mAP@0,5 0.521 | 3052 prop/h
Rtd  umbral 0.39 | recall 0.808 [0.786, 0.829] | prec 0.673 | F1 0.734 | mAP@0,3 0.601 | mAP@0,5 0.499 | 3008 prop/h


### Tabla principal, tabla de ejes y macros de resultado


In [11]:
def ic(valores: list[float]) -> str:
    return f"[{number(valores[0], 3)}; {number(valores[1], 3)}]"


filas_global = []
for k in ORDEN:
    r = R[k]
    filas_global.append(
        f"{NOMBRE[k]} & {number(r['umbral'], 2)} & {number(r['val'].recall, 3)} & "
        f"{number(r['test'].recall, 3)} & {ic(r['ic']['recall'])} & "
        f"{number(r['test'].precision, 3)} & {number(r['f1'], 3)} & "
        f"{number(r['map'][LOOSE_IOU], 3)} & {number(r['map'][MATCH_IOU], 3)} & "
        f"{number(r['map'][STRICT_IOU], 3)}"
    )
escribir_tabla("global", filas_global)

filas_ejes = []
for k in ORDEN:
    e = R[k]["ejes"]
    filas_ejes.append(
        f"{NOMBRE[k]} & {number(e.recall, 3)} & {number(e.precision, 3)} & "
        f"{number(e.median_iou, 3)} & {number(e.recall_strict, 3)} & "
        f"{number(e.class_correct, 3)} & {number(e.species_correct, 3)}"
    )
escribir_tabla("ejes", filas_ejes)

for k in ORDEN:
    r, e = R[k], R[k]["ejes"]
    put(**{
        f"r_threshold_{k.lower()}": number(r["umbral"], 2),
        f"r_recall_{k.lower()}": number(r["test"].recall, 3),
        f"r_recall_ci_{k.lower()}": ic(r["ic"]["recall"]),
        f"r_precision_{k.lower()}": number(r["test"].precision, 3),
        f"r_fone_{k.lower()}": number(r["f1"], 3),
        f"r_map_{k.lower()}": number(r["map"][MATCH_IOU], 3),
        f"r_map_strict_{k.lower()}": number(r["map"][STRICT_IOU], 3),
        f"r_map_loose_{k.lower()}": number(r["map"][LOOSE_IOU], 3),
        f"r_iou_{k.lower()}": number(e.median_iou, 3),
        f"r_class_ok_{k.lower()}": number(e.class_correct, 3),
        f"r_species_ok_{k.lower()}": number(e.species_correct, 3),
        f"r_strict_{k.lower()}": number(e.recall_strict, 3),
        f"r_load_{k.lower()}": number(r["propuestas_hora"]),
        f"r_fptp_{k.lower()}": number(r["fp_tp"], 2),
        f"r_above_{k.lower()}": number(r["test"].n_above),
        f"r_tp_{k.lower()}": number(r["test"].n_tp),
    })

# ¿Separa la métrica primaria a alguna pareja? Dos intervalos que se solapan no separan.
def solapan(a: str, b: str) -> bool:
    x, y = R[a]["ic"]["recall"], R[b]["ic"]["recall"]
    return x[0] <= y[1] and y[0] <= x[1]


pares_separados = [
    (a, b) for i, a in enumerate(ORDEN) for b in ORDEN[i + 1 :] if not solapan(a, b)
]
mejor_primaria = max(ORDEN, key=lambda k: R[k]["test"].recall)
mejor_secundaria = max(ORDEN, key=lambda k: R[k]["map"][MATCH_IOU])
put(
    r_separated_pairs=(
        "ninguna" if not pares_separados
        else ", ".join(f"{NOMBRE[a]}--{NOMBRE[b]}" for a, b in pares_separados)
    ),
    r_n_separated=str(len(pares_separados)),
    r_best_primary=NOMBRE[mejor_primaria],
    r_best_secondary=NOMBRE[mejor_secundaria],
    r_recall_spread=number(
        max(R[k]["test"].recall for k in ORDEN) - min(R[k]["test"].recall for k in ORDEN), 3
    ),
    r_map_spread=number(
        max(R[k]["map"][MATCH_IOU] for k in ORDEN) - min(R[k]["map"][MATCH_IOU] for k in ORDEN), 3
    ),
    r_class_ok_min=number(min(R[k]["ejes"].class_correct for k in ORDEN), 3),
    r_class_ok_max=number(max(R[k]["ejes"].class_correct for k in ORDEN), 3),
    r_species_ok_min=number(min(R[k]["ejes"].species_correct for k in ORDEN), 3),
)
print("parejas que la métrica primaria separa:", pares_separados or "ninguna")
print("mejor primaria:", mejor_primaria, "| mejor secundaria:", mejor_secundaria)


tablas/global.tex: 4 filas
tablas/ejes.tex: 4 filas
parejas que la métrica primaria separa: ninguna
mejor primaria: Yol | mejor secundaria: Yol


### Lectura por clase

El `recall` por clase sí exige la etiqueta correcta. Las dos clases de ventana van
marcadas: su caja ocupa la ventana entera, de modo que detectarlas es clasificar la
ventana y no entran en la mAP.


In [12]:
from evaluation.protocol import class_rows  # noqa: E402

por_clase = {
    k: {f.name: f for f in class_rows(
        R[k]["emparejado_test"],
        R[k]["test_boxes"],
        volcados[(k, "test")].truth,
        R[k]["umbral"],
        NOMBRES_CLASE,
        VENTANA,
    )}
    for k in ORDEN
}

n_gt = {n: por_clase["Det"][n].n_gt for n in NOMBRES_CLASE}
orden_clases = sorted(NOMBRES_CLASE, key=lambda n: -n_gt[n])
filas_clase = []
for nombre in orden_clases:
    marca = "*" if NOMBRES_CLASE.index(nombre) in VENTANA else ""
    valores = [None if (v := por_clase[k][nombre].recall) is None else round(v, 3) for k in ORDEN]
    mejor = max(v for v in valores if v is not None)
    celdas = [
        "---" if v is None else (f"\\textbf{{{number(v, 3)}}}" if v == mejor else number(v, 3))
        for v in valores
    ]
    filas_clase.append(
        f"{tt(nombre)}{marca} & {number(n_gt[nombre])} & " + " & ".join(celdas)
    )

promedios = {k: np.mean([por_clase[k][NOMBRES_CLASE[c]].recall for c in DETECCION]) for k in ORDEN}
filas_clase.append(
    "\\midrule \\textbf{Promedio por clase} & & "
    + " & ".join(f"\\textbf{{{number(promedios[k], 3)}}}" for k in ORDEN)
)
filas_clase.append(
    "Agregado (micro) & "
    + number(sum(n_gt.values()))
    + " & "
    + " & ".join(number(R[k]["test"].recall, 3) for k in ORDEN)
)
escribir_tabla("porclase", filas_clase)

escasas = [n for n in NOMBRES_CLASE if n_gt[n] <= 60]
flojas = [n for n in NOMBRES_CLASE if max(por_clase[k][n].recall for k in ORDEN) < 0.5]
put(
    r_macro_det=number(promedios["Det"], 3),
    r_macro_frc=number(promedios["Frc"], 3),
    r_macro_yol=number(promedios["Yol"], 3),
    r_macro_rtd=number(promedios["Rtd"], 3),
    r_best_macro=NOMBRE[max(ORDEN, key=lambda k: promedios[k])],
    r_hard_classes=", ".join(tt(n) for n in flojas),
    r_n_hard_classes=str(len(flojas)),
    r_n_scarce=str(len(escasas)),
    r_share_two_window=percent(sum(n_gt[NOMBRES_CLASE[c]] for c in VENTANA) / sum(n_gt.values())),
)
print("clases que ningún modelo pasa de 0,5:", flojas)
print({k: round(v, 3) for k, v in promedios.items()})


tablas/porclase.tex: 27 filas
clases que ningún modelo pasa de 0,5: ['ac/gc', 'sm/sc']
{'Det': np.float64(0.636), 'Frc': np.float64(0.634), 'Yol': np.float64(0.694), 'Rtd': np.float64(0.665)}


### Cobertura frente a carga de revisión

Un umbral compartido no es un punto de operación compartido: la confianza de cada
arquitectura vive en su propia escala. La magnitud comparable es cuántos eventos
recupera cada modelo cuando a todos se les permite proponer la misma cantidad de cajas
por hora de audio. Se barre el umbral de cada modelo y se lee su curva a la misma
abscisa.


In [13]:
def curva_carga(clave: str) -> pd.DataFrame:
    m = R[clave]["emparejado_test"]
    orden = np.argsort(-m.scores.numpy(), kind="stable")
    encontrados = np.cumsum(m.found.numpy()[orden])
    propuestas = np.arange(1, len(orden) + 1)
    return pd.DataFrame(
        {"carga": propuestas / HORAS_TEST, "recall": encontrados / m.n_gt,
         "score": m.scores.numpy()[orden]}
    )


curvas_carga = {k: curva_carga(k) for k in ORDEN}

# Para pgfplots: una rejilla común y pocos puntos; la curva es monótona y suave.
rejilla = np.unique(np.concatenate([np.linspace(200, 12000, 120), [3000.0]]))
tabla_carga = pd.DataFrame({"carga": rejilla})
for k in ORDEN:
    c = curvas_carga[k]
    tabla_carga[k] = np.interp(rejilla, c["carga"], c["recall"], left=0.0, right=np.nan)
escribir_dat("carga", tabla_carga.dropna(how="all", subset=ORDEN))


def cobertura_a(clave: str, carga: float) -> float | None:
    c = curvas_carga[clave]
    return None if carga > c["carga"].iloc[-1] else float(np.interp(carga, c["carga"], c["recall"]))


PRESUPUESTOS = [1500, 2000, 3000, 4000, 6000, 9000]
filas_carga = []
for presupuesto in PRESUPUESTOS:
    valores = [None if v is None else round(v, 3) for v in (cobertura_a(k, presupuesto) for k in ORDEN)]
    mejor = max(v for v in valores if v is not None)
    celdas = [
        "---" if v is None else (f"\\textbf{{{number(v, 3)}}}" if v == mejor else number(v, 3))
        for v in valores
    ]
    filas_carga.append(number(presupuesto) + " & " + " & ".join(celdas))
techos = {k: float(curvas_carga[k]["recall"].iloc[-1]) for k in ORDEN}
cargas_techo = {k: float(curvas_carga[k]["carga"].iloc[-1]) for k in ORDEN}
filas_carga.append(
    "\\midrule Techo (sin umbral) & "
    + " & ".join(
        f"\\textbf{{{number(techos[k], 3)}}}" if techos[k] == max(techos.values())
        else number(techos[k], 3)
        for k in ORDEN
    )
)
filas_carga.append(
    "Carga en el techo & "
    + " & ".join(
        f"\\textbf{{{number(cargas_techo[k])}}}" if cargas_techo[k] == min(cargas_techo.values())
        else number(cargas_techo[k])
        for k in ORDEN
    )
)
escribir_tabla("carga", filas_carga)

carga_operacion = np.mean([R[k]["propuestas_hora"] for k in ORDEN])
cobertura_operacion = {k: cobertura_a(k, 3000) for k in ORDEN}
put(
    r_budget_common=number(3000),
    r_mean_load=number(carga_operacion),
    **{f"r_cov3k_{k.lower()}": number(cobertura_operacion[k], 3) for k in ORDEN},
    **{f"r_ceiling_{k.lower()}": number(techos[k], 3) for k in ORDEN},
    **{f"r_ceiling_load_{k.lower()}": number(cargas_techo[k]) for k in ORDEN},
    r_best_at_budget=NOMBRE[max(ORDEN, key=lambda k: cobertura_operacion[k])],
    r_cheapest_ceiling=NOMBRE[min(ORDEN, key=lambda k: cargas_techo[k])],
)
print("cobertura a 3 000 propuestas/h:", {k: round(v, 3) for k, v in cobertura_operacion.items()})
print("techo:", {k: round(v, 3) for k, v in techos.items()})
print("carga en el techo:", {k: round(v) for k, v in cargas_techo.items()})


data/carga.dat: 121x5
tablas/carga.tex: 8 filas
cobertura a 3 000 propuestas/h: {'Det': 0.799, 'Frc': 0.793, 'Yol': 0.807, 'Rtd': 0.807}
techo: {'Det': 0.932, 'Frc': 0.953, 'Yol': 0.912, 'Rtd': 0.94}
carga en el techo: {'Det': 133421, 'Frc': 84093, 'Yol': 7528, 'Rtd': 109197}


### Barrido del umbral del modelo seleccionado

El umbral no se hereda: se elige en validación como el de mayor `recall` entre los que
cumplen la precisión mínima. La serie muestra las dos curvas en validación y dónde
cae el umbral elegido.


In [14]:
GANADOR = max(ORDEN, key=lambda k: (R[k]["map"][MATCH_IOU], -R[k]["propuestas_hora"]))
d_val = volcados[(GANADOR, "val")]
m_val = match(equalize(d_val.predictions, MAX_DETECTIONS), d_val.truth, d_val.n_images)
rejilla_umbral = np.round(np.arange(0.01, 1.0, 0.01), 2)
barrido = pd.DataFrame(
    [
        {
            "umbral": u,
            "recall": operating_point(m_val, float(u)).recall,
            "precision": operating_point(m_val, float(u)).precision or 1.0,
        }
        for u in rejilla_umbral
    ]
)
escribir_dat("barrido", barrido)
put(
    r_winner=NOMBRE[GANADOR],
    r_winner_run=tt(CORRIDAS[GANADOR]),
    r_winner_key=GANADOR,
)
print("ganador:", GANADOR, NOMBRE[GANADOR], "umbral", R[GANADOR]["umbral"])


data/barrido.dat: 99x3
ganador: Yol YOLO26s umbral 0.11


### Distribución del IoU de los pares emparejados

Mide encuadre, no detección: sólo entran las predicciones que ya acertaron un evento.


In [15]:
from torchvision.ops import box_convert, box_iou  # noqa: E402

from evaluation.metrics import assignments, overlaps  # noqa: E402


def ious_emparejados(clave: str) -> np.ndarray:
    cajas = R[clave]["test_boxes"]
    sobre = cajas.select(torch.arange(int((cajas.scores >= R[clave]["umbral"]).sum())))
    verdad = volcados[(clave, "test")].truth
    pares = assignments(overlaps(sobre, verdad, class_aware=False), MATCH_IOU)
    p = torch.tensor([a for a, _ in pares])
    t = torch.tensor([b for _, b in pares])
    return box_iou(
        box_convert(sobre.boxes[p], "cxcywh", "xyxy"),
        box_convert(verdad.boxes[t], "cxcywh", "xyxy"),
    ).diagonal().numpy()


BORDES = np.round(np.arange(0.3, 1.001, 0.05), 3)
hist = {"iou": (BORDES[:-1] + BORDES[1:]) / 2}
ious = {}
for k in ORDEN:
    ious[k] = ious_emparejados(k)
    conteo, _ = np.histogram(ious[k], bins=BORDES)
    hist[k] = conteo / conteo.sum()
escribir_dat("iou", pd.DataFrame(hist))

put(**{f"r_iou_strict_share_{k.lower()}": percent((ious[k] >= STRICT_IOU).mean()) for k in ORDEN})
print({k: (round(float(np.median(v)), 3), round(float((v >= 0.5).mean()), 3)) for k, v in ious.items()})


data/iou.dat: 14x5
{'Det': (0.766, 0.907), 'Frc': (0.768, 0.909), 'Yol': (0.792, 0.922), 'Rtd': (0.797, 0.92)}


### Matriz de confusión del modelo seleccionado

Entre los pares ya emparejados sin mirar la clase: de lo que el modelo encontró, qué
etiqueta le puso. Son 25x25 celdas, de modo que es la única tabla de este capítulo que
se dibuja como imagen.


In [16]:
def pares_etiquetados(clave: str) -> tuple[np.ndarray, np.ndarray]:
    cajas = R[clave]["test_boxes"]
    sobre = cajas.select(torch.arange(int((cajas.scores >= R[clave]["umbral"]).sum())))
    verdad = volcados[(clave, "test")].truth
    pares = assignments(overlaps(sobre, verdad, class_aware=False), MATCH_IOU)
    predicha = np.array([int(sobre.labels[a]) for a, _ in pares])
    anotada = np.array([int(verdad.labels[b]) for _, b in pares])
    return anotada, predicha


anotada, predicha = pares_etiquetados(GANADOR)
n = len(NOMBRES_CLASE)
matriz = np.zeros((n, n))
for a, p in zip(anotada, predicha):
    matriz[a, p] += 1
fila_suma = matriz.sum(axis=1, keepdims=True)
normalizada = np.divide(matriz, fila_suma, out=np.zeros_like(matriz), where=fila_suma > 0)

fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN * 0.92))
ax.grid(False)
im = ax.imshow(normalizada, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n), NOMBRES_CLASE, rotation=90, fontsize=6, family="monospace")
ax.set_yticks(range(n), NOMBRES_CLASE, fontsize=6, family="monospace")
ax.set_xlabel("etiqueta predicha")
ax.set_ylabel("etiqueta anotada")
for i in range(n):
    for j in range(n):
        if normalizada[i, j] >= 0.01:
            ax.text(
                j, i, f"{normalizada[i, j]:.2f}".lstrip("0"),
                ha="center", va="center", fontsize=4.4,
                color="white" if normalizada[i, j] > 0.55 else "black",
            )
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label="fracción de la fila")
guardar(fig, "matriz_confusion")

fuera_diagonal = matriz.sum() - np.trace(matriz)
confusiones = sorted(
    (
        (matriz[i, j], NOMBRES_CLASE[i], NOMBRES_CLASE[j])
        for i in range(n) for j in range(n) if i != j and matriz[i, j] > 0
    ),
    reverse=True,
)
put(
    r_confusion_pairs=number(matriz.sum()),
    r_confusion_off=number(fuera_diagonal / matriz.sum(), 3),
    r_top_confusion=f"{tt(confusiones[0][1])} $\\to$ {tt(confusiones[0][2])}",
    r_top_confusion_n=number(confusiones[0][0]),
    r_top_confusion_share=percent(confusiones[0][0] / fuera_diagonal),
    r_same_species_share=percent(
        sum(c for c, a, b in confusiones if a.split("/")[0] == b.split("/")[0]) / fuera_diagonal
    ),
)
print("confusiones más frecuentes:", confusiones[:5])


matriz_confusion.png
confusiones más frecuentes: [(np.float64(17.0), 'ac/bc', 'sm/cc'), (np.float64(14.0), 'sb/pcc', 'sb/ppc'), (np.float64(10.0), 'sm/hic', 'sb/spc'), (np.float64(8.0), 'lw/trino', 'lw/vc'), (np.float64(8.0), 'ac/sc', 'sm/sc')]


### Costo de inferencia

Medido sobre las mismas ventanas de prueba y en la misma GPU, incluyendo el
preprocesamiento de la cabeza, la supresión de no máximos y el tope de cajas. Se
reporta en minutos de GPU por hora de audio, que es la magnitud que OE3 necesita.


In [17]:
from models.registry import load_checkpoint  # noqa: E402
from utils.boxes import postprocess  # noqa: E402

DISPOSITIVO = "cuda:1" if torch.cuda.is_available() else "cpu"
N_CRONO, LOTE_CRONO = 512, 16
imagenes_test = torch.load(
    PROJECT_DIR / "data/processed/test.pt", map_location="cpu", weights_only=False
)["images"]
muestra = imagenes_test[:N_CRONO]


def cronometrar(clave: str) -> float:
    cargado = load_checkpoint(RUNS / CORRIDAS[clave] / "best.pt", DISPOSITIVO)
    modelo = cargado.model.eval()
    with torch.no_grad():  # una pasada de calentamiento que no se cuenta
        modelo.detect(muestra[:LOTE_CRONO].to(DISPOSITIVO), SCORE_FLOOR)
    if DISPOSITIVO.startswith("cuda"):
        torch.cuda.synchronize()
    inicio = time.time()
    with torch.no_grad():
        for i in range(0, N_CRONO, LOTE_CRONO):
            for det in modelo.detect(muestra[i : i + LOTE_CRONO].to(DISPOSITIVO), SCORE_FLOOR):
                postprocess(det, modelo.nms_iou, MAX_DETECTIONS)
    if DISPOSITIVO.startswith("cuda"):
        torch.cuda.synchronize()
    segundos = time.time() - inicio
    del modelo, cargado
    torch.cuda.empty_cache()
    # minutos de GPU por hora de audio: N_CRONO ventanas cubren N_CRONO * clip_hop_s de audio
    return (segundos / 60) / (N_CRONO * P.clip_hop_s / 3600)


GPU_NOMBRE = torch.cuda.get_device_name(int(DISPOSITIVO.split(":")[1])) if DISPOSITIVO != "cpu" else "CPU"
costo = {k: cronometrar(k) for k in ORDEN}
for k in ORDEN:
    put(**{f"r_gpu_min_{k.lower()}": number(costo[k], 2),
           f"r_realtime_{k.lower()}": number(60 / costo[k])})
put(
    r_gpu_name=GPU_NOMBRE,
    r_timing_windows=number(N_CRONO),
    r_cheapest=NOMBRE[min(ORDEN, key=lambda k: costo[k])],
    r_cost_ratio=number(max(costo.values()) / min(costo.values()), 1),
)
print({k: round(v, 3) for k, v in costo.items()}, GPU_NOMBRE)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Overriding model.yaml nc=80 with nc=25


Overriding model.yaml nc=80 with nc=25


WARNING ⚠️ no model scale passed. Assuming scale='l'.


WARNING ⚠️ no model scale passed. Assuming scale='l'.


{'Det': 0.302, 'Frc': 0.874, 'Yol': 0.12, 'Rtd': 0.32} NVIDIA RTX A5500


### Ejemplos cualitativos

Seis ventanas de prueba con la anotación del equipo (verde, continua) y la predicción
del modelo seleccionado a su umbral (celeste, discontinua). Se eligen a propósito: dos
escenas densas, dos aciertos limpios, un falso positivo de alta confianza y un evento
que el modelo no encontró. Mostrar sólo aciertos debilita el capítulo.


In [18]:
from matplotlib.patches import Rectangle  # noqa: E402

from utils.audio import mel_to_db  # noqa: E402

cajas_test = volcados[(GANADOR, "test")]
sobre_umbral = R[GANADOR]["test_boxes"]
sobre_umbral = sobre_umbral.select(
    torch.arange(int((sobre_umbral.scores >= R[GANADOR]["umbral"]).sum()))
)
pred_por_ventana: dict[int, list[int]] = {}
for fila, ventana in enumerate(sobre_umbral.image_ids.tolist()):
    pred_por_ventana.setdefault(ventana, []).append(fila)

verdad = cajas_test.truth
gt_por_ventana: dict[int, list[int]] = {}
for fila, ventana in enumerate(verdad.image_ids.tolist()):
    gt_por_ventana.setdefault(ventana, []).append(fila)

emparejadas = {a for a, _ in assignments(overlaps(sobre_umbral, verdad, False), MATCH_IOU)}
gt_encontradas = {b for _, b in assignments(overlaps(sobre_umbral, verdad, False), MATCH_IOU)}

fuentes_test = json.loads((PROJECT_DIR / "data/processed/test_sources.json").read_text())
grabacion_de = lambda v: fuentes_test["recording_of_window"][v]  # noqa: E731

densas = sorted(gt_por_ventana, key=lambda v: -len(gt_por_ventana[v]))
limpias = sorted(
    (v for v, filas in gt_por_ventana.items()
     if 2 <= len(filas) <= 4 and all(f in gt_encontradas for f in filas)),
    key=lambda v: -len(gt_por_ventana[v]),
)
perdidas = sorted(
    (v for v, filas in gt_por_ventana.items() if any(f not in gt_encontradas for f in filas)),
    key=lambda v: -len(gt_por_ventana[v]),
)
falsos = sorted(
    (
        v
        for v, filas in pred_por_ventana.items()
        if any(f not in emparejadas and float(sobre_umbral.scores[f]) > 0.8 for f in filas)
    ),
    key=lambda v: -max(float(sobre_umbral.scores[f]) for f in pred_por_ventana[v]),
)

# Un panel por situación y, dentro de cada una, grabaciones distintas: dos ventanas vecinas
# de la misma grabación comparten la mitad del audio y no enseñan nada nuevo.
SITUACIONES = [
    ("escena densa", densas, 2),
    ("todo recuperado", limpias, 2),
    ("evento no recuperado", perdidas, 1),
    ("falso positivo de alta confianza", falsos, 1),
]
elegidas: list[tuple[str, int]] = []
usadas: set[int] = set()
for titulo, grupo, cupo in SITUACIONES:
    tomadas = 0
    for v in grupo:
        if tomadas == cupo:
            break
        if grabacion_de(v) not in usadas:
            elegidas.append((titulo, v))
            usadas.add(grabacion_de(v))
            tomadas += 1
assert len(elegidas) == 6, elegidas

fig, axes = plt.subplots(2, 3, figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN * 0.66),
                         constrained_layout=True)
for ax, (situacion, ventana) in zip(axes.ravel(), elegidas, strict=True):
    ax.imshow(
        mel_to_db(imagenes_test[ventana, 0]).numpy(),
        origin="lower", aspect="auto", cmap="magma", extent=(0, 1, 0, 1),
    )
    vistas = set()
    for fila in gt_por_ventana.get(ventana, []):
        cx, cy, w, h = verdad.boxes[fila].tolist()
        ax.add_patch(Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False,
                               edgecolor="#D7FF3A", linewidth=1.0))
        nombre = NOMBRES_CLASE[int(verdad.labels[fila])]
        if nombre not in vistas:  # una clase se rotula una vez por panel
            vistas.add(nombre)
            ax.text(0.02, 0.98 - 0.08 * (len(vistas) - 1), nombre, color="#D7FF3A",
                    fontsize=5.4, va="top", family="monospace", transform=ax.transAxes,
                    bbox=dict(facecolor="black", alpha=0.45, pad=0.9, edgecolor="none"))
    mejor_score = {}
    for fila in pred_por_ventana.get(ventana, []):
        cx, cy, w, h = sobre_umbral.boxes[fila].tolist()
        ax.add_patch(Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False, edgecolor=CELESTE,
                               linewidth=1.0, linestyle=(0, (3, 2))))
        nombre = NOMBRES_CLASE[int(sobre_umbral.labels[fila])]
        mejor_score[nombre] = max(mejor_score.get(nombre, 0.0), float(sobre_umbral.scores[fila]))
    for i, (nombre, score) in enumerate(sorted(mejor_score.items(), key=lambda x: -x[1])[:3]):
        ax.text(0.98, 0.03 + 0.08 * i, f"{nombre} {score:.2f}", color=CELESTE, fontsize=5.4,
                ha="right", va="bottom", family="monospace", transform=ax.transAxes,
                bbox=dict(facecolor="black", alpha=0.45, pad=0.9, edgecolor="none"))
    ax.set_title(situacion, fontsize=6.2)
    ax.set(xticks=[], yticks=[])
    ax.grid(False)
guardar(fig, "detecciones_cualitativas")
put(r_qualitative_windows=str(len(elegidas)))
print("ventanas cualitativas:", elegidas)


detecciones_cualitativas.png
ventanas cualitativas: [('escena densa', 2742), ('escena densa', 2787), ('todo recuperado', 152), ('todo recuperado', 241), ('evento no recuperado', 261), ('falso positivo de alta confianza', 7580)]


## 4. RE2.4 — el modelo final se carga y produce una tabla de Raven

La verificación es doble: que `load_checkpoint` reconstruya el modelo y cargue el
`state_dict` en modo estricto (ninguna clave de más ni de menos), y que `detect.py`
sobre un WAV deje un archivo con exactamente las columnas de Raven.


In [19]:
from data.raven import SCORE, raven_table  # noqa: E402
from inference.catalog import DETECTIONS_SUFFIX  # noqa: E402

final = load_checkpoint(RUNS / CORRIDAS[GANADOR] / "best.pt", "cpu")
reconstruido = set(final.model.state_dict())
guardado = set(
    torch.load(RUNS / CORRIDAS[GANADOR] / "best.pt", map_location="cpu", weights_only=False)[
        "state_dict"
    ]
)
assert reconstruido == guardado, "el checkpoint no carga sin claves incompatibles"
put(
    r_final_tensors=number(len(guardado)),
    r_final_mb=number((RUNS / CORRIDAS[GANADOR] / "best.pt").stat().st_size / 1e6),
    r_final_epoch=str(corridas[GANADOR]["mejor_epoca"]),
    r_final_classes=str(len(final.labels)),
    r_final_threshold=number(final.operating_point, 2),
)
print(f"{len(guardado)} tensores, idénticos; umbral {final.operating_point}")


Overriding model.yaml nc=80 with nc=25


708 tensores, idénticos; umbral 0.11


### Demostración sobre una grabación de prueba

Se elige, entre las grabaciones de la partición de prueba, una corta con varias clases
anotadas; el modelo nunca la vio. Se ejecuta por línea de comandos, como en el paquete
portable, y se comprueba que la tabla que deja tiene las columnas de Raven.


In [20]:
grabaciones_test = fuentes_test["recordings"]
anotadas = experimento.groupby("audio_path")["label"].agg(["nunique", "size"])
candidatas = anotadas.loc[anotadas.index.isin(grabaciones_test)]
candidatas = candidatas.loc[(candidatas["nunique"] >= 2) & (candidatas["size"].between(4, 30))]

import soundfile as sf  # noqa: E402

duraciones_cand = {p: sf.info(p).duration for p in candidatas.index}
candidatas = candidatas.assign(segundos=pd.Series(duraciones_cand))
candidatas = candidatas.loc[candidatas["segundos"].between(20, 70)].sort_values(
    "nunique", ascending=False
)
DEMO = candidatas.index[0]
print(DEMO, candidatas.iloc[0].to_dict())


/home/fcandia/tesis-primate/data/raw/bolivian_squirrel_monkey__SB/20240516_083055.wav {'nunique': 5.0, 'size': 14.0, 'segundos': 35.22467120181406}


In [21]:
import shutil  # noqa: E402
import tempfile  # noqa: E402

trabajo = Path(tempfile.mkdtemp(prefix="demo_re24_"))
copia = trabajo / Path(DEMO).name
shutil.copy(DEMO, copia)

comando = [
    "uv", "run", "python", "src/detect.py", str(copia),
    "--model", str(RUNS / CORRIDAS[GANADOR] / "best.pt"), "--device", "cpu",
]
proceso = subprocess.run(
    ["/home/fcandia/venv/bin/uv", *comando[1:]],
    cwd=PROJECT_DIR, capture_output=True, text=True, check=True,
)
transcripcion = (
    "$ " + " ".join(comando).replace(str(PROJECT_DIR) + "/", "").replace(str(trabajo), "demo")
    + "\n"
    + (proceso.stdout + proceso.stderr).replace(str(trabajo), "demo").strip()
)
(FIG / "detect_stdout.txt").write_text(transcripcion + "\n")
print(transcripcion)

salida = copia.with_suffix(copia.suffix + DETECTIONS_SUFFIX)
if not salida.exists():
    salida = next(trabajo.glob("*" + DETECTIONS_SUFFIX))
tabla = pd.read_csv(salida, sep="\t")
shutil.copy(salida, FIG / "demo.detections.txt")
print(tabla.columns.tolist())
print(tabla.head())

ESPERADAS = [
    "Selection", "View", "Channel", "Begin Time (s)", "End Time (s)",
    "Low Freq (Hz)", "High Freq (Hz)", "Species", "Call type", "Score",
]
assert tabla.columns.tolist() == ESPERADAS, tabla.columns.tolist()

filas_salida = []
for _, fila in tabla.sort_values("Begin Time (s)").head(8).iterrows():
    filas_salida.append(
        f"{int(fila['Selection'])} & {number(fila['Begin Time (s)'], 2)} & "
        f"{number(fila['End Time (s)'], 2)} & {number(fila['Low Freq (Hz)'])} & "
        f"{number(fila['High Freq (Hz)'])} & {fila['Species']} & {fila['Call type']} & "
        f"{number(fila['Score'], 2)}"
    )
escribir_tabla("salida", filas_salida)


$ uv run python src/detect.py demo/20240516_083055.wav --model runs/yolo26s_v3/best.pt --device cpu
Loading detection engine...
Overriding model.yaml nc=80 with nc=25
[2026-09-30 23:26:31] [INFO] [models.registry] yolo | 25 clases | época 16 | umbral 0.11 | /home/fcandia/tesis-primate/runs/yolo26s_v3/best.pt
[2026-09-30 23:26:31] [INFO] [detect] yolo26s_v3 on cpu | score >= 0.11 | 1 recordings
[2026-09-30 23:26:35] [INFO] [detect] [1/1] 20240516_083055.wav: 55 detections in 4 s -> 20240516_083055.detections.txt
[2026-09-30 23:26:35] [INFO] [detect] Done. Open each audio in Raven with its .detections.txt table
['Selection', 'View', 'Channel', 'Begin Time (s)', 'End Time (s)', 'Low Freq (Hz)', 'High Freq (Hz)', 'Species', 'Call type', 'Score']
   Selection           View  Channel  Begin Time (s)  End Time (s)  \
0          1  Spectrogram 1        1        0.344397      1.152154   
1          2  Spectrogram 1        1        1.511785      1.574157   
2          3  Spectrogram 1        1  

### La demostración frente a las anotaciones del equipo

Es un ejemplo, no una métrica: las métricas son las de RE2.3. Sirve para ver que la
salida cae donde el analista dibujó y en el formato que abre su herramienta.


In [22]:
from utils.audio import hz_to_y  # noqa: E402

anotaciones_demo = experimento.loc[experimento["audio_path"] == DEMO].sort_values("begin_time_s")
duracion_demo = sf.info(DEMO).duration


def a_cajas(df, inicio, fin, bajo, alto):
    return np.stack(
        [df[inicio].to_numpy(), df[fin].to_numpy(),
         hz_to_y(df[bajo].to_numpy(), P), hz_to_y(df[alto].to_numpy(), P)], axis=-1
    )


gt_demo = a_cajas(anotaciones_demo, "begin_time_s", "end_time_s", "low_freq_hz", "high_freq_hz")
pred_demo = a_cajas(tabla, "Begin Time (s)", "End Time (s)", "Low Freq (Hz)", "High Freq (Hz)")


def iou_1d(a, b):
    inter = np.maximum(
        0, np.minimum(a[:, None, 1], b[None, :, 1]) - np.maximum(a[:, None, 0], b[None, :, 0])
    ) * np.maximum(
        0, np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 2], b[None, :, 2])
    )
    area = lambda x: (x[:, 1] - x[:, 0]) * (x[:, 3] - x[:, 2])  # noqa: E731
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


solape = iou_1d(gt_demo, pred_demo)
encontradas = int((solape.max(axis=1) >= MATCH_IOU).sum())
aciertos = int((solape.max(axis=0) >= MATCH_IOU).sum())
etiqueta_ok = 0
for j in range(len(pred_demo)):
    i = int(solape[:, j].argmax())
    if solape[i, j] >= MATCH_IOU:
        par = f"{tabla.iloc[j]['Species']}/{tabla.iloc[j]['Call type']}".lower()
        etiqueta_ok += int(par == anotaciones_demo.iloc[i]["label"])

from utils.audio import mel_spectrogram  # noqa: E402

audio, _ = sf.read(DEMO, dtype="float32")
espectro = mel_to_db(
    mel_spectrogram(P)(torch.from_numpy(audio).unsqueeze(0))
).squeeze(0).numpy()

# A 35 s por el ancho del texto las cajas quedan ilegibles. Se dibuja el tramo de
# VENTANA_DEMO segundos que concentra más anotaciones; la cuenta sobre la grabación
# entera sigue siendo la que reportan las macros.
VENTANA_DEMO = 12.0
inicio_demo = max(
    (0.0, *gt_demo[:, 0]),
    key=lambda t0: int(((gt_demo[:, 0] >= t0) & (gt_demo[:, 1] <= t0 + VENTANA_DEMO)).sum()),
)
inicio_demo = float(min(inicio_demo, max(0.0, duracion_demo - VENTANA_DEMO)))
fin_demo = min(inicio_demo + VENTANA_DEMO, duracion_demo)
tramas_por_s = espectro.shape[1] / duracion_demo
recorte = espectro[:, int(inicio_demo * tramas_por_s) : int(fin_demo * tramas_por_s)]


def dentro(cajas):
    return cajas[(cajas[:, 1] > inicio_demo) & (cajas[:, 0] < fin_demo)]


gt_tramo, pred_tramo = dentro(gt_demo), dentro(pred_demo)

fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 2.6), constrained_layout=True)
ax.imshow(recorte, origin="lower", aspect="auto", cmap="magma",
          extent=(inicio_demo, fin_demo, 0, 1))
for (b, e, lo, hi) in gt_tramo:
    ax.add_patch(Rectangle((b, lo), e - b, hi - lo, fill=False, edgecolor="#D7FF3A",
                           linewidth=1.3))
for (b, e, lo, hi) in pred_tramo:
    ax.add_patch(Rectangle((b, lo), e - b, hi - lo, fill=False, edgecolor=CELESTE,
                           linewidth=1.1, linestyle=(0, (3, 2))))
ax.legend(
    handles=[
        Rectangle((0, 0), 1, 1, fill=False, edgecolor="#D7FF3A", linewidth=1.3,
                  label=f"anotaciones del equipo ({len(gt_tramo)})"),
        Rectangle((0, 0), 1, 1, fill=False, edgecolor=CELESTE, linewidth=1.1,
                  linestyle=(0, (3, 2)), label=f"detecciones del modelo ({len(pred_tramo)})"),
    ],
    loc="upper right", fontsize=6.5, framealpha=0.85,
)
ax.set(xlabel="tiempo (s)", ylabel="frecuencia (mel)", xlim=(inicio_demo, fin_demo),
       yticks=[0, 0.5, 1])
ax.grid(False)
guardar(fig, "demo_grabacion")

put(
    r_demo_file=tt(Path(DEMO).name),
    r_demo_seconds=number(duracion_demo, 1),
    r_demo_annotations=str(len(anotaciones_demo)),
    r_demo_classes=", ".join(tt(c) for c in sorted(anotaciones_demo["label"].unique())),
    r_demo_detections=str(len(tabla)),
    r_demo_found=str(encontradas),
    r_demo_hits=str(aciertos),
    r_demo_same_label=str(etiqueta_ok),
    r_demo_columns=str(len(ESPERADAS)),
    r_demo_window=number(VENTANA_DEMO, 0),
    r_demo_window_start=number(inicio_demo, 1),
    r_demo_window_annotations=str(len(gt_tramo)),
    r_demo_window_detections=str(len(pred_tramo)),
)
print(f"{len(tabla)} detecciones | {encontradas}/{len(anotaciones_demo)} anotaciones encontradas "
      f"| {aciertos} aciertos | {etiqueta_ok} con la misma etiqueta")
shutil.rmtree(trabajo)


demo_grabacion.png
55 detecciones | 13/14 anotaciones encontradas | 15 aciertos | 13 con la misma etiqueta


## 5. RE2.5 — el repositorio

La URL, el commit con el que se generaron estas cifras y los artefactos publicados.


In [23]:
def git(*args: str) -> str:
    return subprocess.run(
        ["git", *args], cwd=PROJECT_DIR, capture_output=True, text=True, check=True
    ).stdout.strip()


put(
    r_repo_url="https://github.com/nhrot-fc/tesis-primate",
    r_commit=tt(git("rev-parse", "--short", "HEAD")),
    r_n_commits=number(int(git("rev-list", "--count", "HEAD"))),
    r_version=tt(
        re.search(r'version\s*=\s*"([^"]+)"', (PROJECT_DIR / "pyproject.toml").read_text()).group(1)
    ),
    r_python=tt(
        re.search(r'requires-python\s*=\s*"([^"]+)"', (PROJECT_DIR / "pyproject.toml").read_text())
        .group(1)
        .replace(">=", "")
    ),
)
print(VALORES["r_commit"], VALORES["r_version"], VALORES["r_n_commits"])


\code{b895fe2} \code{0.2.5} 123


In [24]:
escribir_macros()


225 macros -> figures/valores.tex
